# Messages  消息
消息是包含以下内容的对象：
- Role角色 - 标识消息类型（例如 system 、 user ）
- Content内容 ——指消息的实际内容（例如文本、图像、音频、文档等）。
- Memtadata元数据 - 可选字段，例如响应信息、消息 ID 和令牌使用情况

## Basic usage  基本用法

In [ ]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

qwen_base_url = os.getenv("QWEN_BASE_URL")
qwen_api_key = os.getenv("QWEN_API_KEY")


qwen_model = ChatOpenAI(
    base_url=qwen_base_url,
    api_key=qwen_api_key,
    model = "glm-4.7",
    streaming=True
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage("你是一个有帮助的助手")
human_msg = HumanMessage("你好")

# Use with chat models
messages = [system_msg, human_msg]
response = qwen_model.invoke(messages)  # Returns AIMessage
response.content

### Text prompts  文本提示
`response = model.invoke("Write a haiku about spring")`

有一个单独的请求/不需要对话记录/希望代码复杂度尽可能低。
### Message prompts  消息提示
通过提供消息对象列表，将消息列表传递给模型。

管理多轮对话/处理多模态内容（图像、音频、文件）/包括系统说明

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是一个有帮助的助手"),
    HumanMessage("你好"),
    AIMessage("用户的名字是xiaoxin")
]
response = qwen_model.invoke(messages)
response

### Dictionary format  字典格式
可以直接以 OpenAI 聊天自动补全格式指定消息。

In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = qwen_model.invoke(messages)
response.content

## Message types  消息类型
### System message 系统消息 
SystemMessage 代表一组初始指令，用于启动模型的行为。您可以使用系统消息来设定基调、定义模型的角色并建立响应准则。

### Human message 人类消息 
HumanMessage 代表用户输入和交互。它们可以包含文本、图像、音频、文件以及任何其他数量的多模态内容 。
#### Text content  文本内容

In [ ]:
response = model.invoke([
  HumanMessage("What is machine learning?")
])

# Using a string is a shortcut for a single HumanMessage
response = model.invoke("What is machine learning?")

#### Message metadata  消息元数据
name 字段的行为因提供商而异——有些提供商将其用于用户识别，有些则忽略它。要查看具体情况，请参阅模型提供商的文档 。

In [ ]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

### AI message AI 消息
AIMessage 表示模型调用的输出。它们可以包含多模态数据、工具调用以及特定于提供商的元数据，您可以稍后访问这些元数据。

Attributes  属性:
- text  文本 string 消息正文内容。
- content  内容 string | dict[] 信息的原始内容。
- content_blocks  内容块 ContentBlock[] 消息的标准化内容块。
- tool_calls  工具调用 dict[] | None 模型发出的工具调用。
- id string 消息的唯一标识符（由 LangChain 自动生成或在提供程序响应中返回）
- usage_metadata  用法元数据 dict | None 消息的使用元数据，其中可能包含令牌计数（如有）。
- response_metadata  响应元数据 ResponseMetadata | None 消息的响应元数据。

不同的提供商对消息类型赋予不同的权重/上下文信息，这意味着有时手动创建一个新的 AIMessage 对象并将其插入到消息历史记录中会很有帮助，就像它来自模型一样。

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("不管怎么样,我都不会在得到你父母的允许之前回答你的问题,我会做的只是鼓励你")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("你能帮我写作业吗?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("这是我作业的一题:2+2等于?")
]

response = qwen_model.invoke(messages)
response.content

In [ ]:
response = qwen_model.invoke("Explain AI")
print(type(response))  # <class 'langchain.messages.AIMessage'>

#### Tool calls  工具调用
当模型进行工具调用时，这些调用会包含在 AIMessage 中：

In [ ]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = qwen_model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

#### Token usage  代币使用
AIMessage 可以在其 usage_metadata 字段中保存令牌计数和其他使用元数据：

In [ ]:
response = qwen_model.invoke("Hello!")
response

#### Streaming and chunks  流媒体和分块
在流式传输过程中，您将收到 AIMessageChunk 对象，这些对象可以组合成一个完整的消息对象：

In [ ]:
chunks = []
full_message = None
for chunk in qwen_model.stream("Hi"):
    chunks.append(chunk)
    print(chunk.text, end="", flush=True) 
    full_message = chunk if full_message is None else full_message + chunk
print(full_message.content)

### Tool message  工具消息
工具可以直接生成 ToolMessage 对象。
- content  内容 string `required` 工具调用的字符串化输出。
- tool_call_id  工具调用 ID string `required` 此消息所响应的工具调用 ID。必须与 AIMessage 中的工具调用 ID 匹配。
- name  姓名 string `required` 被调用的工具的名称。
- artifact  dict 未发送给模型但可通过编程方式访问的其他数据。
  > artifact 字段存储补充数据，这些数据不会发送到模型，但可以通过编程方式访问。这对于存储原始结果、调试信息或用于下游处理的数据非常有用，而不会使模型的上下文变得混乱。
```python
from langchain.messages import ToolMessage

# Sent to model
message_content = "It was the best of times, it was the worst of times."

# Artifact available downstream
artifact = {"document_id": "doc_123", "page": 0}

tool_message = ToolMessage(
    content=message_content,
    tool_call_id="call_123",
    name="search_books",
    artifact=artifact,
)
```


In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = qwen_model.invoke(messages)  # Model processes the result
response.content

## Message content  消息内容
消息具有一个弱类型 content 属性，支持字符串和无类型对象列表（例如字典）。这使得 LangChain 聊天模型能够直接支持提供者原生结构，例如多模态内容和其他数据。

In [ ]:
from langchain.messages import HumanMessage

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

### Standard content blocks  标准内容块
消息对象实现了一个 content_blocks 属性，该属性会将 content 属性延迟解析为标准的、类型安全的表示形式。例如， ChatAnthropic 或 ChatOpenAI 生成的消息将包含相应提供商格式的 thinking 或 reasoning 块，但可以延迟解析为一致的 ReasoningContentBlock 表示形式：

In [ ]:
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {
            "type": "reasoning",
            "id": "rs_abc123",
            "summary": [
                {"type": "summary_text", "text": "summary 1"},
                {"type": "summary_text", "text": "summary 2"},
            ],
        },
        {"type": "text", "text": "...", "id": "msg_abc123"},
    ],
    response_metadata={"model_provider": "openai"}
)
message.content_blocks

### Multimodal  多模态
多模态是指处理不同形式数据的能力，例如文本、音频、图像和视频。
1. image

In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "file_id": "file-abc123"},
    ]
}

2. PDF

In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "url": "https://example.com/path/to/document.pdf"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {
            "type": "file",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "application/pdf",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "file_id": "file-abc123"},
    ]
}

In [ ]:
3. Audio 音频

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {
            "type": "audio",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "audio/wav",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {"type": "audio", "file_id": "file-abc123"},
    ]
}

4. video

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {
            "type": "video",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "video/mp4",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {"type": "video", "file_id": "file-abc123"},
    ]
}

### Content block reference  内容块引用

Content Blocks 是 LangChain v1 中用于标准化模型输出的结构化数据格式。  
所有内容块都包含一个 `type` 字段，用于区分不同类型的数据块。

---

#### 基础结构

一个 message 的结构化内容通常是：

```python
message.content_blocks
```

每个 block 是一个字典（TypedDict），格式类似：

```json
{
  "type": "<block_type>",
  ...
}
```

---

##### 1️⃣ Text Block

标准文本输出。

```json
{
  "type": "text",
  "text": "Hello world"
}
```

字段说明：

- `type`: 固定为 `"text"`
- `text`: 文本内容
- 可选字段：
  - `id`
  - `annotations`
  - `index`

---

##### 2️⃣ Tool Call

模型发起工具调用时生成。

```json
{
  "type": "tool_call",
  "name": "get_weather",
  "args": {
    "city": "Tokyo"
  },
  "id": "call_123"
}
```

字段说明：

- `type`: `"tool_call"`
- `name`: 工具名称
- `args`: 工具参数（dict）
- `id`: 工具调用唯一 ID

---

##### 3️⃣ Tool Call Chunk（流式工具调用片段）

流式输出中，工具调用会被拆分成多个 chunk。

```json
{
  "type": "tool_call_chunk",
  "name": "get_weather",
  "args": "{\"city\": \"Tok",
  "id": "call_123",
  "index": 0
}
```

字段说明：

- `type`: `"tool_call_chunk"`
- `name`: 工具名称
- `args`: 部分 JSON 字符串
- `id`: 调用 ID
- `index`: chunk 顺序

---

##### 4️⃣ Invalid Tool Call

当工具调用解析失败时生成。

```json
{
  "type": "invalid_tool_call",
  "name": "get_weather",
  "args": {},
  "error": "Invalid JSON format"
}
```

字段说明：

- `type`: `"invalid_tool_call"`
- `name`: 工具名称
- `args`: 原始参数
- `error`: 错误信息

---

##### 5️⃣ Server Tool Call

服务端执行的工具调用。

```json
{
  "type": "server_tool_call",
  "name": "database_query",
  "args": {
    "sql": "SELECT * FROM users"
  },
  "id": "server_call_1"
}
```

---

##### 6️⃣ Server Tool Call Chunk

流式服务端工具调用片段。

```json
{
  "type": "server_tool_call_chunk",
  "name": "database_query",
  "args": "{ \"sql\": \"SELECT",
  "id": "server_call_1",
  "index": 0
}
```

---

##### 7️⃣ Server Tool Result

服务端工具执行结果。

```json
{
  "type": "server_tool_result",
  "tool_call_id": "server_call_1",
  "id": "result_1",
  "status": "success",
  "output": {
    "rows": 10
  }
}
```

字段说明：

- `type`: `"server_tool_result"`
- `tool_call_id`: 对应调用 ID
- `id`: 结果 ID
- `status`: `"success"` 或 `"error"`
- `output`: 执行结果数据

---

##### 8️⃣ Image Block

图片内容。

```json
{
  "type": "image",
  "url": "https://example.com/image.png"
}
```

或

```json
{
  "type": "image",
  "base64": "<base64_string>",
  "mime_type": "image/png"
}
```

字段：

- `url` 或 `base64`
- `mime_type`
- `id`（可选）

---

##### 9️⃣ Audio Block

音频内容。

```json
{
  "type": "audio",
  "url": "https://example.com/audio.mp3"
}
```

---

##### 🔟 Video Block

视频内容。

```json
{
  "type": "video",
  "url": "https://example.com/video.mp4"
}
```

---

##### 1️⃣1️⃣ File Block

文件内容（PDF / 文档等）。

```json
{
  "type": "file",
  "url": "https://example.com/file.pdf",
  "mime_type": "application/pdf"
}
```

---

##### 1️⃣2️⃣ Plain Text Block

纯文本文件类型。

```json
{
  "type": "text-plain",
  "text": "Raw document content",
  "mime_type": "text/plain"
}
```

---

##### 1️⃣3️⃣ Non-Standard Block

非标准或 provider 自定义结构。

```json
{
  "type": "non_standard",
  "value": {
    "provider_specific_field": "value"
  }
}
```

---

#### 总结

| 类型 | 作用 |
|------|------|
| text | 普通文本 |
| tool_call | 工具调用 |
| tool_call_chunk | 流式工具调用片段 |
| invalid_tool_call | 无效工具调用 |
| server_tool_call | 服务端工具调用 |
| server_tool_result | 服务端执行结果 |
| image | 图片 |
| audio | 音频 |
| video | 视频 |
| file | 文件 |
| text-plain | 文本文件 |
| non_standard | 非标准扩展数据 |

---

#### 工程理解

- `.content` = 原始结构数据  
- `.text` = 仅提取 text 类型拼接后的字符串  
- Agent / Tool 解析必须依赖结构化 content blocks  
- 多模态和流式输出都基于 block 结构实现  